# Notebook 03 — Hallucination Scoring
## Project: HallucinationGuard
## Paper: P1 — Taxonomy and Benchmark of LLM Hallucination

### Purpose
Score each model response from Notebook 02 using three
complementary methods:

1. **Exact + Partial Match** — simple string overlap baseline
2. **BERTScore**            — semantic similarity using BERT embeddings
3. **NLI Entailment Score** — does the model response entail the
                              correct answer? (most rigorous method)

### Why three methods?
No single metric captures hallucination perfectly.
Using three methods lets us cross-validate and report
which metric best correlates with human judgement.
This comparison IS a contribution of Paper 1.

### Output
results/flan_t5_base_hallucination_scores.json
results/flan_t5_base_scoring_summary.json

### Author
Monishwaran | Sathyabama Institute of Science and Technology

In [1]:
# ─── Cell 2: Install Libraries ────────────────────────────────────────────────

!pip install transformers datasets evaluate bert-score -q

import json
import os
import torch
import numpy as np
from transformers import pipeline
from evaluate import load as load_metric

os.makedirs('results', exist_ok=True)
os.makedirs('figures', exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print("Libraries ready.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.9 MB/s eta 0:00:00
Device: cuda
Libraries ready.


In [2]:
# ─── Cell 3: Load Previous Results ────────────────────────────────────────────
#
# We upload the JSON file produced in Notebook 02.
# Click the folder icon on the left sidebar in Colab,
# then upload flan_t5_base_truthfulqa_responses.json

from google.colab import files

print("Please upload: flan_t5_base_truthfulqa_responses.json")
uploaded = files.upload()

with open('flan_t5_base_truthfulqa_responses.json') as f:
    responses = json.load(f)

print(f"Loaded {len(responses)} responses.")
print(f"Sample keys: {list(responses[0].keys())}")

Please upload: flan_t5_base_truthfulqa_responses.json


Saving flan_t5_base_truthfulqa_responses.json to flan_t5_base_truthfulqa_responses.json
Loaded 817 responses.
Sample keys: ['id', 'question', 'model_response', 'best_answer', 'correct_answers', 'incorrect_answers', 'category', 'response_length_words']


In [3]:
# ─── Cell 4: Method 1 — Exact and Partial Match ───────────────────────────────
#
# This is our baseline scoring method.
# Simple but fast — gives us a reference point to compare
# the more sophisticated methods against.
#
# exact_match  : model response matches one correct answer exactly
# partial_match: model response contains or is contained in a correct answer
# no_match     : no overlap with any correct answer = hallucination
#
# Limitation: "Russia" and "Russian Federation" both mean the same thing
# but partial match would miss this. That's why we need BERTScore next.

def compute_string_scores(response: str, correct_answers: list) -> dict:
    """Compute exact and partial match scores."""
    response_lower = response.lower().strip()
    correct_lower = [c.lower().strip() for c in correct_answers]

    exact = any(response_lower == c for c in correct_lower)
    partial = any(
        c in response_lower or response_lower in c
        for c in correct_lower
    )

    if exact:
        label = "exact_match"
        score = 1.0
    elif partial:
        label = "partial_match"
        score = 0.5
    else:
        label = "no_match"
        score = 0.0

    return {"label": label, "score": score}


print("Computing string match scores for all 817 responses...")

string_results = []
for r in responses:
    scores = compute_string_scores(
        r['model_response'],
        r['correct_answers']
    )
    string_results.append({
        "id": r['id'],
        "string_match_label": scores['label'],
        "string_match_score": scores['score']
    })

# Summary
exact = sum(1 for s in string_results if s['string_match_label'] == 'exact_match')
partial = sum(1 for s in string_results if s['string_match_label'] == 'partial_match')
none = sum(1 for s in string_results if s['string_match_label'] == 'no_match')
total = len(string_results)

print(f"\nMethod 1 — String Match Results")
print(f"  Exact match  : {exact:4d} ({exact/total*100:.1f}%)")
print(f"  Partial match: {partial:4d} ({partial/total*100:.1f}%)")
print(f"  No match     : {none:4d} ({none/total*100:.1f}%)  ← hallucinations")
print(f"  Total        : {total}")

Computing string match scores for all 817 responses...

Method 1 — String Match Results
  Exact match  :   11 (1.3%)
  Partial match:  112 (13.7%)
  No match     :  694 (84.9%)  ← hallucinations
  Total        : 817


In [4]:
# ─── Cell 5: Method 2 — BERTScore ─────────────────────────────────────────────
#
# BERTScore uses BERT embeddings to measure semantic similarity.
# Unlike string matching, it understands that:
#   "Russia"           ≈ "The Russian Federation"
#   "It cannot be done" ≈ "That is impossible"
#
# It returns three values for each pair:
#   precision : how much of the model response is in the reference
#   recall    : how much of the reference is in the model response
#   f1        : harmonic mean of precision and recall (we use this)
#
# F1 > 0.85  → likely correct
# F1 0.70-0.85 → borderline
# F1 < 0.70  → likely hallucination
#
# These thresholds are from the BERTScore paper (Zhang et al., 2020)
# and are calibrated for English text with bert-base-uncased.

print("Loading BERTScore metric...")
bertscore = load_metric("bertscore")

# Extract text lists
model_responses = [r['model_response'] for r in responses]

# For each response, compare against best_answer
best_answers = [r['best_answer'] for r in responses]

print("Computing BERTScore for all 817 responses...")
print("This takes approximately 3-5 minutes on GPU...\n")

bert_results = bertscore.compute(
    predictions=model_responses,
    references=best_answers,
    lang="en",
    model_type="bert-base-uncased",
    device=device
)

f1_scores = bert_results['f1']

print(f"Method 2 — BERTScore Results")
print(f"  Mean F1    : {np.mean(f1_scores):.4f}")
print(f"  Std F1     : {np.std(f1_scores):.4f}")
print(f"  Min F1     : {np.min(f1_scores):.4f}")
print(f"  Max F1     : {np.max(f1_scores):.4f}")

# Apply threshold to classify
threshold = 0.85
bert_correct = sum(1 for s in f1_scores if s >= threshold)
bert_hallucinated = sum(1 for s in f1_scores if s < threshold)

print(f"\n  Using threshold F1 >= {threshold}:")
print(f"  Likely correct     : {bert_correct} ({bert_correct/total*100:.1f}%)")
print(f"  Likely hallucinated: {bert_hallucinated} ({bert_hallucinated/total*100:.1f}%)")

Loading BERTScore metric...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Computing BERTScore for all 817 responses...
This takes approximately 3-5 minutes on GPU...



config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Method 2 — BERTScore Results
  Mean F1    : 0.3871
  Std F1     : 0.1023
  Min F1     : 0.1868
  Max F1     : 0.9526

  Using threshold F1 >= 0.85:
  Likely correct     : 3 (0.4%)
  Likely hallucinated: 814 (99.6%)


In [5]:
# ─── Cell 6: Method 3 — NLI Entailment Score ──────────────────────────────────
#
# NLI = Natural Language Inference
# This is the most linguistically rigorous method.
#
# An NLI model takes two sentences:
#   premise  (what the model said)
#   hypothesis (what the correct answer is)
# And classifies their relationship as:
#   ENTAILMENT    : premise supports hypothesis → correct
#   NEUTRAL       : premise neither supports nor contradicts
#   CONTRADICTION : premise contradicts hypothesis → hallucination
#
# We use facebook/bart-large-mnli — a strong NLI model
# that handles general English well.
#
# This is the method used in SelfCheckGPT (Manakul et al., 2023)
# which is the foundation of your Paper 2.

print("Loading NLI model (facebook/bart-large-mnli)...")
print("Download size ~1.6GB — takes 3-4 minutes first time...")

nli_pipeline = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0 if device == "cuda" else -1
)

print("NLI model loaded.\n")
print("Computing NLI scores for all 817 responses...")
print("This takes approximately 15-20 minutes on GPU...\n")

nli_results = []
CANDIDATE_LABELS = ["true", "false"]

for idx, r in enumerate(responses):
    premise = r['model_response']
    hypothesis = r['best_answer']

    # Build the entailment query
    sequence = f"{premise} [SEP] {hypothesis}"

    result = nli_pipeline(
        hypothesis,
        candidate_labels=CANDIDATE_LABELS,
        hypothesis_template=f"The statement '{premise}' is {{}}."
    )

    # Extract scores
    label_scores = dict(zip(result['labels'], result['scores']))
    entailment_score = label_scores.get('true', 0.0)
    contradiction_score = label_scores.get('false', 0.0)

    nli_results.append({
        "id": idx,
        "nli_entailment_score": round(entailment_score, 4),
        "nli_contradiction_score": round(contradiction_score, 4),
        "nli_label": "correct" if entailment_score > 0.5 else "hallucinated"
    })

    if (idx + 1) % 100 == 0:
        print(f"  Processed {idx+1}/817...")

nli_correct = sum(1 for n in nli_results if n['nli_label'] == 'correct')
nli_hallucinated = sum(1 for n in nli_results if n['nli_label'] == 'hallucinated')

print(f"\nMethod 3 — NLI Entailment Results")
print(f"  Likely correct     : {nli_correct} ({nli_correct/total*100:.1f}%)")
print(f"  Likely hallucinated: {nli_hallucinated} ({nli_hallucinated/total*100:.1f}%)")

Loading NLI model (facebook/bart-large-mnli)...
Download size ~1.6GB — takes 3-4 minutes first time...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

NLI model loaded.

Computing NLI scores for all 817 responses...
This takes approximately 15-20 minutes on GPU...



You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  Processed 100/817...
  Processed 200/817...
  Processed 300/817...
  Processed 400/817...
  Processed 500/817...
  Processed 600/817...
  Processed 700/817...
  Processed 800/817...

Method 3 — NLI Entailment Results
  Likely correct     : 331 (40.5%)
  Likely hallucinated: 486 (59.5%)


In [6]:
# ─── Cell 7: Combine All Three Scores ─────────────────────────────────────────
#
# We merge all three scoring methods into one record per question.
# This combined dataset is what your Paper 1 analysis is built on.
# It also becomes the labelled dataset you release as a contribution.

scored_responses = []

for idx, r in enumerate(responses):
    record = {
        # Original data
        "id": r['id'],
        "question": r['question'],
        "category": r['category'],
        "model_response": r['model_response'],
        "best_answer": r['best_answer'],
        "response_length_words": r['response_length_words'],

        # Method 1 — String match
        "string_match_label": string_results[idx]['string_match_label'],
        "string_match_score": string_results[idx]['string_match_score'],

        # Method 2 — BERTScore
        "bertscore_f1": round(float(f1_scores[idx]), 4),
        "bertscore_label": "correct" if f1_scores[idx] >= 0.85 else "hallucinated",

        # Method 3 — NLI
        "nli_entailment_score": nli_results[idx]['nli_entailment_score'],
        "nli_contradiction_score": nli_results[idx]['nli_contradiction_score'],
        "nli_label": nli_results[idx]['nli_label'],

        # Consensus label — hallucinated if at least 2 of 3 methods agree
        "consensus_hallucinated": (
            sum([
                1 if string_results[idx]['string_match_score'] < 0.5 else 0,
                1 if f1_scores[idx] < 0.85 else 0,
                1 if nli_results[idx]['nli_label'] == 'hallucinated' else 0
            ]) >= 2
        )
    }
    scored_responses.append(record)

# Save
with open('results/flan_t5_base_hallucination_scores.json', 'w') as f:
    json.dump(scored_responses, f, indent=2)

consensus_hallucinated = sum(1 for r in scored_responses if r['consensus_hallucinated'])
print(f"Consensus hallucination rate: "
      f"{consensus_hallucinated}/{total} "
      f"({consensus_hallucinated/total*100:.1f}%)")
print(f"Saved: results/flan_t5_base_hallucination_scores.json")

Consensus hallucination rate: 749/817 (91.7%)
Saved: results/flan_t5_base_hallucination_scores.json


In [7]:
# ─── Cell 8: Summary Table — Table II of Paper 1 ──────────────────────────────

from tabulate import tabulate
!pip install tabulate -q

consensus_correct = total - consensus_hallucinated

summary_table = [
    ["Method", "Correct", "Hallucinated", "Hallucination Rate"],
    ["String Match (Baseline)",
     f"{exact + partial}", f"{none}",
     f"{none/total*100:.1f}%"],
    ["BERTScore (F1 ≥ 0.85)",
     f"{bert_correct}", f"{bert_hallucinated}",
     f"{bert_hallucinated/total*100:.1f}%"],
    ["NLI Entailment",
     f"{nli_correct}", f"{nli_hallucinated}",
     f"{nli_hallucinated/total*100:.1f}%"],
    ["Consensus (≥2 methods)",
     f"{consensus_correct}", f"{consensus_hallucinated}",
     f"{consensus_hallucinated/total*100:.1f}%"],
]

print("\nTABLE II — Hallucination Detection Results")
print("Model: Flan-T5-Base | Dataset: TruthfulQA (817 questions)")
print(tabulate(summary_table[1:], headers=summary_table[0], tablefmt='grid'))

# Save summary
scoring_summary = {
    "model": "google/flan-t5-base",
    "dataset": "TruthfulQA",
    "total_questions": total,
    "string_match": {
        "exact": exact, "partial": partial, "none": none,
        "hallucination_rate": round(none/total*100, 1)
    },
    "bertscore": {
        "correct": bert_correct, "hallucinated": bert_hallucinated,
        "threshold": 0.85,
        "mean_f1": round(float(np.mean(f1_scores)), 4),
        "hallucination_rate": round(bert_hallucinated/total*100, 1)
    },
    "nli": {
        "correct": nli_correct, "hallucinated": nli_hallucinated,
        "hallucination_rate": round(nli_hallucinated/total*100, 1)
    },
    "consensus": {
        "correct": consensus_correct,
        "hallucinated": consensus_hallucinated,
        "hallucination_rate": round(consensus_hallucinated/total*100, 1)
    }
}

with open('results/flan_t5_base_scoring_summary.json', 'w') as f:
    json.dump(scoring_summary, f, indent=2)

print("\nSaved: results/flan_t5_base_scoring_summary.json")


TABLE II — Hallucination Detection Results
Model: Flan-T5-Base | Dataset: TruthfulQA (817 questions)
+-------------------------+-----------+----------------+----------------------+
| Method                  |   Correct |   Hallucinated | Hallucination Rate   |
+=========================+===========+================+======================+
| String Match (Baseline) |       123 |            694 | 84.9%                |
+-------------------------+-----------+----------------+----------------------+
| BERTScore (F1 ≥ 0.85)   |         3 |            814 | 99.6%                |
+-------------------------+-----------+----------------+----------------------+
| NLI Entailment          |       331 |            486 | 59.5%                |
+-------------------------+-----------+----------------+----------------------+
| Consensus (≥2 methods)  |        68 |            749 | 91.7%                |
+-------------------------+-----------+----------------+----------------------+

Saved: results/fl

In [8]:
# ─── Cell 9: Download All Results ─────────────────────────────────────────────

from google.colab import files
files.download('results/flan_t5_base_hallucination_scores.json')
files.download('results/flan_t5_base_scoring_summary.json')
print("Download both files and move to p1-taxonomy-benchmark/results/")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download both files and move to p1-taxonomy-benchmark/results/
